# 01 · Data tour

A first look at the FishCast training data, for new members. Run from the repository root with `data/krillcast_merged.csv` in place (see `data/README.md`).

You will see: what a haul is, where and when the hauls were taken, how many catch nothing, how hotspots are defined, and how the data is split into training, validation and test years.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / 'fishcast').exists():
    ROOT = ROOT.parent  # running from notebooks/
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fishcast.data import load_krill_dataset

DATA = ROOT / 'data' / 'krillcast_merged.csv'
raw = pd.read_csv(DATA, parse_dates=['DATE'])
raw.head()

## 1. What is in a row

Each row is one **haul**: a scientific net sample at a place and date. `NUMBER_OF_KRILL_UNDER_1M2` is krill caught per square metre of sea surface. The seven capitalised columns after it are environmental conditions matched from a climate model (see `scripts/data/README.md`).

In [ ]:
print(f"{len(raw):,} hauls, {raw.DATE.min():%Y} to {raw.DATE.max():%Y}")
print(f"Hauls with no krill: {(raw.NUMBER_OF_KRILL_UNDER_1M2 == 0).mean():.0%}")
raw.describe().T[['mean', 'min', 'max']].round(2)

## 2. Where and when

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4))
a.scatter(raw.LONGITUDE, raw.LATITUDE, s=2, alpha=0.3)
a.set(xlabel='Longitude', ylabel='Latitude', title='Haul locations')
raw.DATE.dt.year.value_counts().sort_index().plot.bar(ax=b, width=1)
b.set(title='Hauls per year', xlabel='')
b.set_xticks(b.get_xticks()[::10])
plt.tight_layout()

Most hauls are in the Atlantic sector around the Antarctic Peninsula (longitude about −70 to −30). Sampling is uneven over time, with a gap in the 1940s–1970s.

## 3. Krill density is very skewed

The model predicts `log(1 + density)` because a few hauls catch thousands of krill per m² while a third catch none.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.5))
raw.NUMBER_OF_KRILL_UNDER_1M2.clip(upper=500).hist(bins=50, ax=a)
a.set(title='Density (clipped at 500)', xlabel='krill per m²')
np.log1p(raw.NUMBER_OF_KRILL_UNDER_1M2).hist(bins=50, ax=b)
b.set(title='log(1 + density): what the model predicts', xlabel='log1p(krill per m²)')
plt.tight_layout()

## 4. Hotspots and the train / validation / test split

`load_krill_dataset` builds the model's inputs. A **hotspot** is a haul in the top 10% of catches for its 5° by 5° area (and it must contain krill). The split is by year: training up to 1998, validation 1998–2004, test 2004–2016. Everything that is fitted (scaling, hotspot cut-offs) uses training rows only.

In [ ]:
b = load_krill_dataset(DATA)
years = b.dataframe['FRACTIONAL_YEAR'].to_numpy()
hot = b.hotspot.numpy().ravel()
for name, idx in [('train', b.train_indices), ('validation', b.val_indices), ('test', b.test_indices)]:
    print(f"{name:>10}: {len(idx):5,} hauls, {years[idx].min():.0f}-{years[idx].max():.0f}, hotspot rate {hot[idx].mean():.1%}")
print('features:', b.feature_columns)
print('coordinates:', b.coord_columns)

## 5. Things to try

- Plot hotspot rate by 5° sector on a map. Which areas are reliably good for krill?
- How does mean `log1p` density change by decade in the Peninsula region?
- Compare SST in hauls with and without krill.

Next: `02_benchmark_results.ipynb`.